# v4 repair 调度与候选保留修复 v1

日期：2026-11-02。新批次 `formal_30_v4_repair_scheduling_v1`，策略 `bounded_prepass_v1`，复用开发样本 TaskManager、MonitoringDashboard。记录在本批首次模型调用前建立。

依据：更长超时诊断 `formal_30_v4_timeout_diagnostic_v1_results_20261102_173627`（272 文件校验通过）完成 2 项但 0 项通过。TaskManager 的 3 个混合 attempt 全被确定性归一化消耗，没有 LLM repair；MonitoringDashboard 第 3 轮引入事件缺失和 initData 冲突，保护门只停止，最终仍引用退化产物。旧结果保持，不重新计分。

## 通用实现修改与预算边界

本次修改的是 Python repair 调度／候选接受逻辑，没有修改 generation 或 repair Prompt，也没有按组件名分支或直接修改冻结 San。

- `migration_pipeline/utils/repair_scheduler.py`：新增显式选用的策略。每次 LLM 之前最多 4 个确定性步骤，每步重新验证；固定点、重复代码、步数上限或候选拒收后交给 LLM。已通过则无需 LLM。最多 3 次 LLM repair，总确定性步数上限 12。
- `migration_pipeline/orchestrator.py`：只在 metadata 指定 bounded_prepass_v1 时进入新策略，legacy 保持旧混合 attempt 定义。
- `migration_pipeline/stages/repair.py`：支持调度器要求的 LLM-only 调用，防止进入该阶段后再次被确定性 return 截断。
- `migration_pipeline/run_batch_pipeline.py`：验证并传递 repair_policy，报告预算单位、LLM 次数、确定性步数与候选保留信息。
- `migration_pipeline/run_pipeline.py`：报告归一化历史、停止原因和拒收文件。

**新 3 次 LLM 调用和旧 3 个混合 attempt 不是等预算口径**。新协议明确记录计算预算变化，比较时同时报告 token 和确定性步数；不能据此回填旧结果。固定点和循环上限约束执行过程，不等于证明所有原归一化规则都幂等。

## 候选拒收行为与离线验证

新策略每次检查事件契约、关键编译／initData 冲突及视觉通过状态退化，不再依赖 Prompt 特定文本标记。坏候选文件和评估仍保留在历史中；LLM 候选拒收后恢复上个候选代码、文件路径与评估，停止并保持失败。确定性候选拒收后交回 LLM。未通过的上一候选不会因此获得成功评分。

离线回放 `python -m migration_pipeline.replay_formal_30_v4_repair_scheduling` 不调用模型：

1. 从冻结 TaskManager 初始代码开始，实际验证／渲染后能够到达 LLM-only 阶段，不再耗尽三个混合 attempt。首次归一化被保守的事件追溯检查拒收（closeModal 的 self-click 包装后追溯不完整），故交给 LLM；这是已知限制，不绕过检查。
2. 单独隔离接受门，重放冻结 MonitoringDashboard 第 2 → 3 轮：第 3 轮被拒，final 文件恢复为第 2 轮，且仍计失败。
3. 重放 BudgetThreshold 初始 → 第一次 repair 的坏模板：坏候选被拒，初始文件被保留；初始静态问题仍在，仍计失败。

详细输入哈希、候选门原因和输出路径：`data/experiments/analysis/formal_30_v4_repair_scheduling_v1_20261102/replay.json`。这些是调度和接受门的验证，不是新的自动迁移通过结果。

## 新批次运行与待办

协议：`data/experiments/protocols/formal_30_v4_repair_scheduling_v1_frozen_20261102/PROTOCOL.md`。继续使用 qwen/qwen-plus、温度 0.2、600／660／660 秒超时。

停止旧服务，在窗口 A 设置环境中的 LLM_API_KEY 后运行 `python -m migration_pipeline.run_formal_30_v4_repair_scheduling --server`；窗口 B 执行一次 `python -m migration_pipeline.run_formal_30_v4_repair_scheduling --execute`。使用独立新输出目录，不加 --force，完成后先冻结结果再诊断。

原三个失败事项仍需后续处理：BudgetThreshold 只验证防退化、未宣称已修好初始数据类型；TipSplitCalculator 事件辅助方法契约与 LocalizationWorkspace 隐藏语言依赖未修复。此策略也不保证 TaskManager／MonitoringDashboard 的所有语义缺陷都能修好。模型新结果尚待操作者执行。之后再决定进一步通用修复与完整 30 项回归，v5 探索性评估继续等待。

由于当前实现已变，旧 30 样本和 v5 的冻结包保持历史证据用途，其 live 校验预期不再匹配新实现；不能修改旧包来消除差异，应为后续运行建新版本。

2026-11-02 预运行检查：238/238 测试通过，其中新增 9 项调度边界测试；离线回放三项检查通过，未调用模型。新协议封存 255 份 payload，263 项 live 哈希校验通过，新的两样本 dry-run 通过。模型批次尚未开始，无新 summary。

## 实际运行结果：两项完成，零项自动通过

运行：2026-11-02 18:32:32–19:04:47（UTC+8）。先冻结后详细诊断，快照 `data/experiments/snapshots/formal_30_v4_repair_scheduling_v1_results_20261102_224614`，283 个文件校验通过；冻结时协议 263 项 live 校验通过。无执行错误、无超时，2/2 完成、0/2 自动通过，没有新的人工交互通过结论。

| 样本 | LLM repair 调用 | 确定性步骤（接受数） | 最终 DOM 相似度 | 停止原因 |
| --- | ---: | ---: | ---: | --- |
| TaskManager | 2 | 2（0） | 0.9889 | repair_candidate_regression_rolled_back |
| MonitoringDashboard | 3 | 3（3） | 0.9605 | max_repair_rounds_reached |

本轮生成 204,452、repair 543,361、总计 747,813 tokens，包含拒收候选消耗。TaskManager 总用时 757.109 秒，MonitoringDashboard 1177.797 秒。新预算为 3 次 LLM repair，旧预算为 3 个混合 attempt，且均重新生成，不作等成本或单因素因果比较。

TaskManager：两次确定性候选均因新增一个无法追溯的 click:closeModal 被拒，随后实际进入 LLM。第一轮 LLM 后剩事件语法、self-click guard、computed 调 isOverdue、getTaskCount 隐藏 tasks 依赖四类问题。第二轮后静态问题减少为两类，但包装后的 closeModal 仍无法被事件追溯器匹配，保护门拒收并停止。最终文件恢复为 TaskManager_repaired.san；其代码、静态和视觉评估均核对匹配第一轮。坏候选仍保留在历史与文件中，计入用量。

第二轮代码有 handleModalBackdropClick(e)，检查 target/currentTarget 并调用 closeModal，但模板未显式传 $event；当前追溯只识别直接绑定及少量别名。不能直接把 unresolved 当成真实功能删除，也不能当成纯误报放行。需继续验证包装的事件传参、条件分支与等价性，并增加正反例测试。调度与回退机制已执行，不代表语义迁移完成。

MonitoringDashboard：三次 LLM 后剩 invalid_san_inline_computed_method_invocation（requestChart computed barWidth 返回函数、模板却按方法调用）及 event_arguments_not_landed（onKPIClick(kpi)、changePeriod(period.value) 参数未保留）。方法改用 this.event.target 取 data-*，不能据初始 DOM 通过证明事件链等价或可用。

详细记录：`data/experiments/analysis/formal_30_v4_repair_scheduling_v1_20261102_224614/REPORT.md`、`summary.json`。复现分析：`python -m migration_pipeline.analyze_formal_30_v4_repair_scheduling --snapshot data/experiments/snapshots/formal_30_v4_repair_scheduling_v1_results_20261102_224614`。

后续计划（本次尚未实施）：先冻结产物离线回放，审查通用事件包装追溯／显式传参及参数化 computed 规则，补正反例，不降低标准、不直接改生成 San。修复须建新版本，再做针对性回归和完整 30 项回归。本次仅新增冻结、诊断和记录工具，未改运行期 Prompt／规则／checker；旧 30 项评分不变，原三个失败项未结案，v5 探索性评估继续等待。

## 后续诊断更正（2026-11-02）

真实 San 事件分发测试确认：裸 handler 由解析器默认传 $event，handler() 才不传参数。因此上一节对未显式写 $event 的怀疑不能作为功能缺失的证据；本批 closeModal 包装被拒的直接原因仍是追溯覆盖缺口。原静态残留和 0/2 评分不变，冻结结果不修改。后续有边界修复及 12 项正反例测试另记 `2026-11-02_formal_30_v4_semantic_closure.ipynb`。